<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/05_GES_Aware_Genomic_RAG_Cell_7A2_V5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GES-RAG Experiment 2 — Cell 7A2 Corrected V5

**Run this notebook before Cell 7A3.**

This notebook performs the frozen T1 feature-reconstruction and model-applicability preflight. It is intentionally fail-closed: if any required check fails, it stops **before** writing Cell 7A2 artifacts.

### Authorized here
- Verify all frozen upstream hashes and the Cell 7A1 authorization
- Reproduce frozen Stage 4 transformations
- Reconstruct T1 features in memory
- Apply already-fitted preprocessing only
- Verify the combined-metadata policy
- Freeze Cell 7A2 audit outputs after a complete PASS

### Not authorized here
- Model fitting, refitting, calibration, or tuning
- Full-GES, No-star, or combined-metadata score generation
- Threshold or reranking-weight selection
- RAG corpus creation, embeddings, prompts, questions, or LLM calls

Use **Runtime → Run all**. Cell 7A3 is authorized only when the final output reports a complete PASS.

In [1]:
# Environment check only — this cell does not install or modify packages.
import sys
import numpy as np
import pandas as pd
import pyarrow
import sklearn
import joblib
import scipy

print("Python       :", sys.version.split()[0])
print("NumPy        :", np.__version__)
print("pandas       :", pd.__version__)
print("PyArrow      :", pyarrow.__version__)
print("scikit-learn :", sklearn.__version__)
print("joblib       :", joblib.__version__)
print("SciPy        :", scipy.__version__)

Python       : 3.12.13
NumPy        : 2.0.2
pandas       : 2.2.2
PyArrow      : 18.1.0
scikit-learn : 1.6.1
joblib       : 1.5.3
SciPy        : 1.16.3


In [2]:

# ==================================================================================================
# EXPERIMENT 2 — STAGE 7A — CELL 7A2 — CORRECTED V5
# FROZEN T1 FEATURE RECONSTRUCTION AND MODEL-APPLICABILITY PREFLIGHT
#
# PURPOSE
#   1. Verify every frozen upstream artifact by SHA-256.
#   2. Reproduce the frozen Stage 4B transformations from Stage 4A raw inputs.
#   3. Reconstruct the six T1 model inputs in memory.
#   4. Verify that the already-fitted Full-GES and No-star pipelines can preprocess all T1 rows.
#   5. Verify the frozen combined-metadata comparator policy.
#   6. Freeze auditable preflight outputs only when every QC check passes.
#
# STRICT BOUNDARY
#   - No model fitting or refitting
#   - No predict(), predict_proba(), decision_function(), or score generation
#   - No threshold or weight optimization
#   - No row-level T1 feature-table persistence
#   - No RAG corpus, embeddings, question set, prompts, or LLM calls
#
# A terminal PASS authorizes Cell 7A3 only.
# ==================================================================================================

from collections import OrderedDict, Counter
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import math
import os
import platform
import re
import sys
import time

import joblib
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy import sparse
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


# --------------------------------------------------------------------------------------------------
# 1. GOOGLE DRIVE AND PROJECT PATHS
# --------------------------------------------------------------------------------------------------

DRIVE_ROOT = Path("/content/drive/MyDrive")

if not DRIVE_ROOT.exists():
    from google.colab import drive
    drive.mount("/content/drive")

if not DRIVE_ROOT.exists():
    raise FileNotFoundError("Google Drive is not mounted at /content/drive/MyDrive.")

ROOT = DRIVE_ROOT / "GES_RAG_Temporal_Study"
NOTEBOOK_NAME = "05_GES_Aware_Genomic_RAG.ipynb"
CELL_ID = "7A2"
PACKAGE_VERSION = "v1"

STAGE4_DATA_DIR = ROOT / "data_processed" / "stage4_ges"
STAGE4_MODEL_DIR = ROOT / "models" / "stage4_ges"
STAGE4_CONFIG_DIR = ROOT / "configs" / "stage4_ges"
STAGE6_CONFIG_DIR = ROOT / "configs" / "stage6_temporal_validation"
STAGE7_CONFIG_DIR = ROOT / "configs" / "stage7_rag"
STAGE7_TABLE_DIR = ROOT / "outputs" / "tables" / "stage7_rag"
STAGE7_QC_DIR = ROOT / "outputs" / "quality_checks" / "stage7_rag"

for directory in [STAGE7_CONFIG_DIR, STAGE7_TABLE_DIR, STAGE7_QC_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


# --------------------------------------------------------------------------------------------------
# 2. EXACT FROZEN INPUT PATHS
# --------------------------------------------------------------------------------------------------

CELL_7A1_MANIFEST = (
    STAGE7_CONFIG_DIR / "cell_7a1_t1_corpus_source_preflight_manifest_v1.json"
)

T1_PARQUET = ROOT / "data_interim" / "t1_rcv_target_genes_harmonized_v1.parquet"
T1_FREEZE_MANIFEST = (
    ROOT / "configs" / "clinvar_t1_target_gene_rcv_extraction_validation_manifest_v1.json"
)

STAGE4A_FEATURE_TABLE = (
    STAGE4_DATA_DIR / "stage4a_t0_ges_baseline_features_v1.parquet"
)
STAGE4A_FEATURE_SPEC = (
    STAGE4_CONFIG_DIR / "stage4a_t0_feature_specification_v1.json"
)
STAGE4A_QC = (
    STAGE4_CONFIG_DIR / "stage4a_t0_feature_qc_report_v1.json"
)
STAGE4A_MANIFEST = (
    STAGE4_CONFIG_DIR / "stage4a_t0_feature_freeze_manifest_v1.json"
)

STAGE4B_TABLE = (
    STAGE4_DATA_DIR / "stage4b_t0_feature_transforms_and_weak_labels_v1.parquet"
)
STAGE4B_TRANSFORM_PARAMETERS = (
    STAGE4_CONFIG_DIR / "stage4b_t0_feature_transform_parameters_v1.json"
)
STAGE4B_WEAK_LABEL_RULES = (
    STAGE4_CONFIG_DIR / "stage4b_weak_label_rules_v1.json"
)
STAGE4B_QC = (
    STAGE4_CONFIG_DIR / "stage4b_weak_label_qc_report_v1.json"
)
STAGE4B_MANIFEST = (
    STAGE4_CONFIG_DIR / "stage4b_weak_label_freeze_manifest_v1.json"
)

FULL_MODEL_PATH = STAGE4_MODEL_DIR / "stage4c_full_ges_logistic_model_v1.joblib"
NO_STAR_MODEL_PATH = STAGE4_MODEL_DIR / "stage4c_no_star_ges_logistic_model_v1.joblib"
STAGE4C_MODEL_SPEC = STAGE4_CONFIG_DIR / "stage4c_ges_model_specification_v1.json"
STAGE4C_QC = STAGE4_CONFIG_DIR / "stage4c_ges_model_qc_report_v1.json"
STAGE4C_MANIFEST = STAGE4_CONFIG_DIR / "stage4c_ges_model_freeze_manifest_v1.json"

STAGE6A_POLICY = STAGE6_CONFIG_DIR / "stage6a_comparator_score_policy_v1.json"


# --------------------------------------------------------------------------------------------------
# 3. EXPECTED HASHES, COUNTS, FEATURES, AND MODEL SETTINGS
# --------------------------------------------------------------------------------------------------

EXPECTED_HASHES = OrderedDict([
    ("cell_7a1_manifest", "84e509d97f01fb8dc0b6ad0c7e24de762e8923b9068fc835bf328105f79e946d"),
    ("t1_parquet", "5713a11bdbf4804758cc011f9b2f302afc91fa1f88c1b178d675c28bb277d37c"),
    ("t1_freeze_manifest", "7eaeff0fee3df96973130f721d6c1f2a02fd9108e85af7b2743cdd7750a4372e"),
    ("stage4a_feature_table", "c100b3781e6801425f622f5d091376abfe0939e48c0a792af32f6eebe6401f16"),
    ("stage4a_feature_spec", "fc00146efe5da9b3fbe740bb42ca99d157252cdefc045650f9e88d54d8fcfa8b"),
    ("stage4a_qc", "4bf72af66aa800fa9f12fdc8598bd06ebc859bb31c97ad97cf7fc11ef1614cf1"),
    ("stage4a_manifest", "2b844ef2dbc0c3e5e57493886a7533587350e1b4b4c76fc9d445ecda8a528fa0"),
    ("stage4b_table", "c2e9e4f96cd1f61d3962e88d28557dfa1221729b8f79f8fd4d2dafb34847bdb8"),
    ("stage4b_transform_parameters", "baeab167e19381138f93c51ba2fa00e4eeed684b36122c80cd2bf9fc4fe4b08d"),
    ("stage4b_weak_label_rules", "3d78e66cea1fed5c75ef1cab1b7cf44d3d3d7bfae50909173bae6c3a0e0bff61"),
    ("stage4b_qc", "03b14efb6d297fd517043c8ff952977715415723f191d079d968c111368d31a9"),
    ("stage4b_manifest", "e766061442e6d4610f661f44a619e41b45dc6363b28f6176d3f9a71f8215c63f"),
    ("stage4c_full_model", "0b4a87b16f768484cbdae168fc226cec5e978521172fb03510bfb1ea3e78fa30"),
    ("stage4c_no_star_model", "6c3fe4fc7fe8fdde7b8f0f0d608c48e66a07945effb8c67c6b98d35e1955257c"),
    ("stage4c_model_spec", "d754c715c990f42cecd64259ca2c420427b9602c669dc7be9e80dee9554f61f6"),
    ("stage4c_qc", "3a1a90d3a8946fd35bde52324a625077f3127ac57f231b20d206d70b2c678ec7"),
    ("stage4c_manifest", "c0d8008a4db80c67f5b1c568ddba3496b2411b29bce0db1eb20e63f26613d4ee"),
    ("stage6a_policy", "dd7e95dc785e77b04c289ef50817b1ddac7516a436d83b5734dbf3cd35b1248d"),
])

ARTIFACT_PATHS = OrderedDict([
    ("cell_7a1_manifest", CELL_7A1_MANIFEST),
    ("t1_parquet", T1_PARQUET),
    ("t1_freeze_manifest", T1_FREEZE_MANIFEST),
    ("stage4a_feature_table", STAGE4A_FEATURE_TABLE),
    ("stage4a_feature_spec", STAGE4A_FEATURE_SPEC),
    ("stage4a_qc", STAGE4A_QC),
    ("stage4a_manifest", STAGE4A_MANIFEST),
    ("stage4b_table", STAGE4B_TABLE),
    ("stage4b_transform_parameters", STAGE4B_TRANSFORM_PARAMETERS),
    ("stage4b_weak_label_rules", STAGE4B_WEAK_LABEL_RULES),
    ("stage4b_qc", STAGE4B_QC),
    ("stage4b_manifest", STAGE4B_MANIFEST),
    ("stage4c_full_model", FULL_MODEL_PATH),
    ("stage4c_no_star_model", NO_STAR_MODEL_PATH),
    ("stage4c_model_spec", STAGE4C_MODEL_SPEC),
    ("stage4c_qc", STAGE4C_QC),
    ("stage4c_manifest", STAGE4C_MANIFEST),
    ("stage6a_policy", STAGE6A_POLICY),
])

EXPECTED_T1_ROWS = 100_920
EXPECTED_T1_COLUMNS = 36
EXPECTED_STAGE4A_ROWS = 71_659
EXPECTED_STAGE4A_COLUMNS = 30
EXPECTED_STAGE4B_ROWS = 71_659
EXPECTED_STAGE4B_COLUMNS = 43

EXPECTED_T1_GENE_COUNTS = {
    "BRCA1": 32_603,
    "BRCA2": 49_221,
    "MLH1": 13_684,
    "EGFR": 5_412,
}

EXPECTED_T1_AXIS_COUNTS = {
    "GermlineClassification": 97_526,
    "OncogenicityClassification": 52,
    "SomaticClinicalImpact": 25,
    "NoClassification": 3_317,
}

EXPECTED_CELL_7A1_DECISION = (
    "PASS_STAGE7A1_T1_CORPUS_SOURCE_VERIFIED_CHECKSUM_PROTECTED_"
    "TOP_LEVEL_AND_NESTED_SCHEMA_INVENTORIED_EVIDENCE_PACKET_"
    "DERIVABILITY_AUDITED_STAGE7A2_PREFLIGHT_ONLY"
)

T1_CUTOFF = pd.Timestamp("2025-12-27")

FULL_FEATURES = [
    "recency_score",
    "recency_missing_flag",
    "submitter_diversity_score",
    "review_confidence",
    "aggregate_conflict_flag",
    "scv_group_entropy_normalized",
]

NO_STAR_FEATURES = [
    "recency_score",
    "recency_missing_flag",
    "submitter_diversity_score",
    "aggregate_conflict_flag",
    "scv_group_entropy_normalized",
]

EXPECTED_MODEL_SETTINGS = {
    "penalty": "l2",
    "C": 1.0,
    "solver": "lbfgs",
    "class_weight": None,
    "max_iter": 2_000,
    "tol": 1e-6,
    "fit_intercept": True,
    "random_state": 42,
}


# --------------------------------------------------------------------------------------------------
# 4. OUTPUT PATHS
# --------------------------------------------------------------------------------------------------

OUTPUTS = OrderedDict([
    ("artifact_inventory", STAGE7_TABLE_DIR / "cell_7a2_frozen_artifact_inventory_v1.csv"),
    ("formula_audit", STAGE7_TABLE_DIR / "cell_7a2_t0_frozen_transformation_formula_audit_v1.csv"),
    ("feature_inventory", STAGE7_TABLE_DIR / "cell_7a2_t1_feature_applicability_inventory_v1.csv"),
    ("model_inventory", STAGE7_TABLE_DIR / "cell_7a2_frozen_model_pipeline_inventory_v1.csv"),
    ("axis_inventory", STAGE7_TABLE_DIR / "cell_7a2_t1_classification_axis_applicability_v1.csv"),
    ("preflight_report", STAGE7_QC_DIR / "cell_7a2_feature_reconstruction_model_applicability_preflight_v1.json"),
    ("qc", STAGE7_QC_DIR / "cell_7a2_feature_reconstruction_model_applicability_qc_v1.json"),
    ("manifest", STAGE7_CONFIG_DIR / "cell_7a2_feature_reconstruction_model_applicability_manifest_v1.json"),
])


# --------------------------------------------------------------------------------------------------
# 5. GENERAL HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    path = Path(path)
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(chunk_size), b""):
            digest.update(block)
    return digest.hexdigest()


def sidecar_path(path: Path) -> Path:
    return Path(str(path) + ".sha256")


def read_sidecar_hash(path: Path) -> str:
    text = Path(path).read_text(encoding="utf-8").strip()
    matches = re.findall(r"\b[a-fA-F0-9]{64}\b", text)
    if not matches:
        raise ValueError(f"No SHA-256 value found in sidecar: {path}")
    return matches[0].lower()


def sidecar_is_valid(path: Path) -> bool:
    path = Path(path)
    sidecar = sidecar_path(path)
    return (
        path.exists()
        and sidecar.exists()
        and read_sidecar_hash(sidecar) == sha256_file(path)
    )


def verify_exact_hash(label: str, path: Path, expected: str) -> str:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required frozen artifact for {label}:\n{path}"
        )
    observed = sha256_file(path)
    if observed != expected:
        raise AssertionError(
            f"SHA-256 mismatch for {label}.\n"
            f"Expected: {expected}\nObserved: {observed}\nPath: {path}"
        )
    return observed


def json_native(value):
    if isinstance(value, dict):
        return {str(key): json_native(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_native(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (pd.Timestamp, datetime)):
        return value.isoformat()
    if isinstance(value, np.ndarray):
        return [json_native(item) for item in value.tolist()]
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if value is pd.NA:
        return None
    if isinstance(value, float) and not np.isfinite(value):
        return None
    return value


def stable_write_bytes(path: Path, payload: bytes) -> str:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(
        f".{path.name}.tmp-{os.getpid()}-{time.time_ns()}"
    )
    temporary.write_bytes(payload)
    proposed_hash = sha256_file(temporary)

    if path.exists():
        existing_hash = sha256_file(path)
        if existing_hash != proposed_hash:
            temporary.unlink(missing_ok=True)
            raise RuntimeError(
                "Refusing to overwrite a nonidentical frozen Cell 7A2 artifact.\n"
                f"Path: {path}\n"
                f"Existing SHA-256: {existing_hash}\n"
                f"Proposed SHA-256: {proposed_hash}"
            )
        temporary.unlink(missing_ok=True)
    else:
        os.replace(temporary, path)

    return sha256_file(path)


def stable_write_json(path: Path, payload: dict) -> str:
    data = (
        json.dumps(
            json_native(payload),
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            allow_nan=False,
        )
        + "\n"
    ).encode("utf-8")
    return stable_write_bytes(path, data)


def stable_write_csv(path: Path, frame: pd.DataFrame) -> str:
    data = frame.to_csv(
        index=False,
        lineterminator="\n",
        float_format="%.12g",
    ).encode("utf-8")
    return stable_write_bytes(path, data)


def write_sidecar(path: Path) -> str:
    path = Path(path)
    payload = f"{sha256_file(path)}  {path.name}\n".encode("utf-8")
    stable_write_bytes(sidecar_path(path), payload)
    return sha256_file(sidecar_path(path))


def resolve_column(columns, aliases, label, required=True):
    exact = {str(column).lower(): str(column) for column in columns}
    for alias in aliases:
        if alias.lower() in exact:
            return exact[alias.lower()]
    if required:
        raise KeyError(
            f"Could not resolve required column '{label}'.\n"
            f"Aliases checked: {aliases}\n"
            f"Available columns: {list(columns)}"
        )
    return None


def parse_json_value(value):
    if value is None or value is pd.NA:
        return None, "missing"
    try:
        if pd.isna(value):
            return None, "missing"
    except Exception:
        pass

    if isinstance(value, (dict, list)):
        return value, "native"

    text = str(value).strip()
    if not text:
        return None, "blank"

    try:
        return json.loads(text), "parsed"
    except json.JSONDecodeError:
        return None, "parse_error"


def normalize_gene(value) -> str:
    allowed = {"BRCA1", "BRCA2", "MLH1", "EGFR"}
    parsed, status = parse_json_value(value)

    if status == "parse_error":
        parsed = [str(value).strip()]
    if isinstance(parsed, str):
        parsed = [parsed]
    if not isinstance(parsed, list):
        return ""

    genes = sorted({
        str(gene).strip().upper()
        for gene in parsed
        if str(gene).strip().upper() in allowed
    })
    return genes[0] if len(genes) == 1 else ""


def boolish_to_float(value):
    if value is None or value is pd.NA:
        return np.nan
    try:
        if pd.isna(value):
            return np.nan
    except Exception:
        pass

    if isinstance(value, (bool, np.bool_)):
        return float(value)
    if isinstance(value, (int, float, np.integer, np.floating)):
        number = float(value)
        return number if number in {0.0, 1.0} else np.nan

    text = str(value).strip().lower()
    if text in {"true", "t", "yes", "y", "1"}:
        return 1.0
    if text in {"false", "f", "no", "n", "0"}:
        return 0.0
    return np.nan


def numeric_dict_sum(mapping):
    if not isinstance(mapping, dict):
        return np.nan, 1

    total = 0.0
    invalid = 0
    for value in mapping.values():
        try:
            number = float(value)
        except (TypeError, ValueError):
            invalid += 1
            continue
        if not np.isfinite(number) or number < 0:
            invalid += 1
            continue
        total += number
    return total, invalid


def normalized_entropy_from_group_counts(value):
    parsed, status = parse_json_value(value)

    if status == "parse_error":
        return np.nan, np.nan, 0, "parse_error"
    if parsed is None:
        return np.nan, np.nan, 0, status

    total, invalid = numeric_dict_sum(parsed)
    if invalid > 0:
        return np.nan, total, 0, "invalid_numeric_value"

    positive_counts = np.asarray(
        [float(item) for item in parsed.values() if float(item) > 0],
        dtype=float,
    )

    if len(positive_counts) == 0:
        return np.nan, total, 0, "no_positive_counts"
    if len(positive_counts) == 1:
        return 0.0, total, 1, "single_group"

    probabilities = positive_counts / positive_counts.sum()
    entropy = -float(np.sum(probabilities * np.log(probabilities)))
    normalized = float(
        np.clip(entropy / math.log(len(positive_counts)), 0.0, 1.0)
    )
    return normalized, total, int(len(positive_counts)), "derived"


def extract_pipeline(artifact, label):
    if isinstance(artifact, Pipeline):
        return artifact, "<top-level>"

    preferred_keys = [
        "pipeline",
        "model_pipeline",
        "fitted_pipeline",
        "sklearn_pipeline",
        "model",
        "estimator",
        "classifier",
    ]

    if isinstance(artifact, dict):
        for key in preferred_keys:
            if key in artifact and isinstance(artifact[key], Pipeline):
                return artifact[key], f"[{key!r}]"

    matches = []
    visited = set()

    def walk(obj, location="<top-level>", depth=0):
        if depth > 8 or id(obj) in visited:
            return
        visited.add(id(obj))

        if isinstance(obj, Pipeline):
            matches.append((location, obj))
            return

        if isinstance(obj, dict):
            for key, item in obj.items():
                walk(item, f"{location}[{key!r}]", depth + 1)
        elif isinstance(obj, (list, tuple)):
            for index, item in enumerate(obj):
                walk(item, f"{location}[{index}]", depth + 1)
        else:
            for attribute in preferred_keys:
                if hasattr(obj, attribute):
                    try:
                        walk(
                            getattr(obj, attribute),
                            f"{location}.{attribute}",
                            depth + 1,
                        )
                    except Exception:
                        pass

    walk(artifact)

    unique = {}
    for location, pipeline in matches:
        unique.setdefault(id(pipeline), (location, pipeline))
    found = list(unique.values())

    if len(found) != 1:
        raise TypeError(
            f"Expected exactly one sklearn Pipeline in {label}; "
            f"found {len(found)}."
        )
    return found[0][1], found[0][0]


def get_exact_component(pipeline, component_type):
    matches = [
        (name, step)
        for name, step in pipeline.steps
        if isinstance(step, component_type)
    ]
    if len(matches) != 1:
        raise AssertionError(
            f"Expected exactly one {component_type.__name__}; "
            f"found {len(matches)}."
        )
    return matches[0]


def transformed_values_are_finite(values) -> bool:
    if sparse.issparse(values):
        return bool(np.isfinite(values.data).all())
    return bool(np.isfinite(np.asarray(values)).all())


def model_settings_match(classifier: LogisticRegression) -> bool:
    return bool(
        classifier.penalty == EXPECTED_MODEL_SETTINGS["penalty"]
        and np.isclose(float(classifier.C), EXPECTED_MODEL_SETTINGS["C"])
        and classifier.solver == EXPECTED_MODEL_SETTINGS["solver"]
        and classifier.class_weight == EXPECTED_MODEL_SETTINGS["class_weight"]
        and int(classifier.max_iter) == EXPECTED_MODEL_SETTINGS["max_iter"]
        and np.isclose(float(classifier.tol), EXPECTED_MODEL_SETTINGS["tol"])
        and bool(classifier.fit_intercept)
            == EXPECTED_MODEL_SETTINGS["fit_intercept"]
        and classifier.random_state == EXPECTED_MODEL_SETTINGS["random_state"]
    )


def find_feature_order(artifact, pipeline, expected_features):
    expected_features = [str(item) for item in expected_features]
    candidate_keys = {
        "feature_columns",
        "feature_names",
        "features",
        "input_features",
        "model_features",
        "predictor_columns",
    }
    candidates = []

    if hasattr(pipeline, "feature_names_in_"):
        candidates.append(
            ("pipeline.feature_names_in_", list(pipeline.feature_names_in_))
        )

    visited = set()

    def walk(obj, location="<top-level>", depth=0):
        if depth > 7 or id(obj) in visited:
            return
        visited.add(id(obj))

        if isinstance(obj, dict):
            for key, value in obj.items():
                key_text = str(key).lower()
                if key_text in candidate_keys and isinstance(
                    value, (list, tuple, np.ndarray, pd.Index)
                ):
                    candidates.append(
                        (f"{location}[{key!r}]", [str(item) for item in value])
                    )
                walk(value, f"{location}[{key!r}]", depth + 1)
        elif isinstance(obj, (list, tuple)):
            for index, value in enumerate(obj):
                walk(value, f"{location}[{index}]", depth + 1)

    walk(artifact)

    exact_matches = [
        (location, values)
        for location, values in candidates
        if values == expected_features
    ]

    if exact_matches:
        return exact_matches[0][1], exact_matches[0][0], True

    if candidates:
        location, values = candidates[0]
        return values, location, False

    return [], "<not-found>", False


def preprocess_without_scoring(pipeline, frame):
    classifier_positions = [
        index
        for index, (_, step) in enumerate(pipeline.steps)
        if isinstance(step, LogisticRegression)
    ]
    if len(classifier_positions) != 1:
        raise AssertionError(
            "Expected exactly one LogisticRegression position in the pipeline."
        )

    classifier_index = classifier_positions[0]
    transformed = frame.copy()

    for step_name, step in pipeline.steps[:classifier_index]:
        if not hasattr(step, "transform"):
            raise AssertionError(
                f"Pre-classifier step '{step_name}' has no transform() method."
            )
        transformed = step.transform(transformed)

    return transformed


# --------------------------------------------------------------------------------------------------
# 6. VERIFY ALL FROZEN INPUT ARTIFACTS AND AUTHORIZATION
# --------------------------------------------------------------------------------------------------

observed_hashes = OrderedDict()

for key, path in ARTIFACT_PATHS.items():
    observed_hashes[key] = verify_exact_hash(
        key, path, EXPECTED_HASHES[key]
    )

if not sidecar_is_valid(CELL_7A1_MANIFEST):
    raise AssertionError("Cell 7A1 manifest sidecar verification failed.")

cell_7a1_payload = json.loads(
    CELL_7A1_MANIFEST.read_text(encoding="utf-8")
)

if cell_7a1_payload.get("terminal_decision") != EXPECTED_CELL_7A1_DECISION:
    raise AssertionError(
        "Cell 7A1 terminal decision does not authorize Cell 7A2."
    )

if (
    cell_7a1_payload.get("next_authorized_cell", {}).get("cell_id")
    != "7A2"
):
    raise AssertionError(
        "Cell 7A1 manifest does not identify Cell 7A2 as the next "
        "authorized cell."
    )

immutable_hashes_before = {
    key: sha256_file(path)
    for key, path in ARTIFACT_PATHS.items()
}


# --------------------------------------------------------------------------------------------------
# 7. LOAD EXACT FROZEN TRANSFORMATION PARAMETERS
# --------------------------------------------------------------------------------------------------

stage4b_transform_payload = json.loads(
    STAGE4B_TRANSFORM_PARAMETERS.read_text(encoding="utf-8")
)

frozen_recency_parameters = stage4b_transform_payload.get(
    "recency_transformation", {}
)
frozen_submitter_parameters = stage4b_transform_payload.get(
    "submitter_diversity_transformation", {}
)

required_frozen_parameter_keys = {
    "maximum_observation_window_days":
        frozen_recency_parameters.get("maximum_observation_window_days"),
    "minimum_log_count":
        frozen_submitter_parameters.get("minimum_log_count"),
    "maximum_log_count":
        frozen_submitter_parameters.get("maximum_log_count"),
}

missing_frozen_parameter_keys = [
    key
    for key, value in required_frozen_parameter_keys.items()
    if value is None
]

if missing_frozen_parameter_keys:
    raise KeyError(
        "The frozen Stage 4B transformation-parameter JSON is missing: "
        f"{missing_frozen_parameter_keys}"
    )

MAXIMUM_OBSERVATION_WINDOW_DAYS = float(
    required_frozen_parameter_keys["maximum_observation_window_days"]
)
SUBMITTER_LOG_MIN = float(
    required_frozen_parameter_keys["minimum_log_count"]
)
SUBMITTER_LOG_MAX = float(
    required_frozen_parameter_keys["maximum_log_count"]
)

if not (
    np.isfinite(MAXIMUM_OBSERVATION_WINDOW_DAYS)
    and MAXIMUM_OBSERVATION_WINDOW_DAYS > 0
    and np.isfinite(SUBMITTER_LOG_MIN)
    and np.isfinite(SUBMITTER_LOG_MAX)
    and SUBMITTER_LOG_MAX > SUBMITTER_LOG_MIN
):
    raise ValueError(
        "Frozen Stage 4B transformation parameters are not numerically valid."
    )


# --------------------------------------------------------------------------------------------------
# 8. VERIFY PARQUET DIMENSIONS AND BUILD FROZEN ARTIFACT INVENTORY
# --------------------------------------------------------------------------------------------------

t1_meta = pq.ParquetFile(T1_PARQUET).metadata
stage4a_meta = pq.ParquetFile(STAGE4A_FEATURE_TABLE).metadata
stage4b_meta = pq.ParquetFile(STAGE4B_TABLE).metadata

artifact_inventory_rows = []

for key, path in ARTIFACT_PATHS.items():
    row = {
        "artifact_key": key,
        "path": str(path),
        "file_name": path.name,
        "suffix": path.suffix.lower(),
        "expected_sha256": EXPECTED_HASHES[key],
        "observed_sha256": observed_hashes[key],
        "hash_verified": observed_hashes[key] == EXPECTED_HASHES[key],
        "bytes": int(path.stat().st_size),
        "sidecar_present": sidecar_path(path).exists(),
        "sidecar_verified": (
            sidecar_is_valid(path) if sidecar_path(path).exists() else False
        ),
        "rows": None,
        "columns": None,
    }

    if path.suffix.lower() == ".parquet":
        metadata = pq.ParquetFile(path).metadata
        row["rows"] = int(metadata.num_rows)
        row["columns"] = int(metadata.num_columns)

    artifact_inventory_rows.append(row)

artifact_inventory = pd.DataFrame(artifact_inventory_rows)


# --------------------------------------------------------------------------------------------------
# 9. REPRODUCE THE FROZEN STAGE 4B TRANSFORMATIONS ON T0
# --------------------------------------------------------------------------------------------------

stage4a_columns = pq.ParquetFile(
    STAGE4A_FEATURE_TABLE
).schema_arrow.names
stage4b_columns = pq.ParquetFile(
    STAGE4B_TABLE
).schema_arrow.names

stage4a_required_columns = [
    "t0_row_order",
    "rcv_accession",
    "recency_days",
    "recency_missing_flag",
    "unique_submitter_count",
    "log1p_unique_submitter_count",
    "aggregate_review_stars",
]

stage4b_required_columns = list(dict.fromkeys([
    "t0_row_order",
    "rcv_accession",
    "recency_score",
    "recency_missing_flag",
    "submitter_diversity_score",
    "review_confidence",
] + FULL_FEATURES))

missing_stage4a_columns = [
    column
    for column in stage4a_required_columns
    if column not in stage4a_columns
]
missing_stage4b_columns = [
    column
    for column in stage4b_required_columns
    if column not in stage4b_columns
]

if missing_stage4a_columns:
    raise KeyError(
        "Frozen Stage 4A table is missing required raw-source columns: "
        f"{missing_stage4a_columns}"
    )

if missing_stage4b_columns:
    raise KeyError(
        "Frozen Stage 4B table is missing required transformed columns: "
        f"{missing_stage4b_columns}"
    )

t0_raw = pd.read_parquet(
    STAGE4A_FEATURE_TABLE,
    columns=stage4a_required_columns,
).copy()

t0 = pd.read_parquet(
    STAGE4B_TABLE,
    columns=stage4b_required_columns,
).copy()

if len(t0_raw) != EXPECTED_STAGE4A_ROWS:
    raise RuntimeError(
        f"Loaded Stage 4A row count is {len(t0_raw):,}; "
        f"expected {EXPECTED_STAGE4A_ROWS:,}."
    )

if len(t0) != EXPECTED_STAGE4B_ROWS:
    raise RuntimeError(
        f"Loaded Stage 4B row count is {len(t0):,}; "
        f"expected {EXPECTED_STAGE4B_ROWS:,}."
    )

t0_raw_row_order = pd.to_numeric(
    t0_raw["t0_row_order"], errors="raise"
).astype("int64")
t0_row_order = pd.to_numeric(
    t0["t0_row_order"], errors="raise"
).astype("int64")

t0_raw_rcv = (
    t0_raw["rcv_accession"].astype("string").str.strip().str.upper()
)
t0_rcv = (
    t0["rcv_accession"].astype("string").str.strip().str.upper()
)

stage4a_row_order_valid = bool(
    t0_raw_row_order.nunique(dropna=False) == EXPECTED_STAGE4A_ROWS
    and np.array_equal(
        t0_raw_row_order.to_numpy(),
        np.arange(EXPECTED_STAGE4A_ROWS, dtype=np.int64),
    )
)

stage4b_row_order_valid = bool(
    t0_row_order.nunique(dropna=False) == EXPECTED_STAGE4B_ROWS
    and np.array_equal(
        t0_row_order.to_numpy(),
        np.arange(EXPECTED_STAGE4B_ROWS, dtype=np.int64),
    )
)

stage4a_stage4b_row_order_alignment_verified = bool(
    np.array_equal(
        t0_raw_row_order.to_numpy(),
        t0_row_order.to_numpy(),
    )
)

stage4a_stage4b_rcv_alignment_verified = bool(
    t0_raw_rcv.notna().all()
    and t0_rcv.notna().all()
    and not t0_raw_rcv.fillna("").eq("").any()
    and not t0_rcv.fillna("").eq("").any()
    and t0_raw_rcv.nunique(dropna=False) == EXPECTED_STAGE4A_ROWS
    and t0_rcv.nunique(dropna=False) == EXPECTED_STAGE4B_ROWS
    and np.array_equal(
        t0_raw_rcv.to_numpy(dtype=str),
        t0_rcv.to_numpy(dtype=str),
    )
)

if not stage4a_row_order_valid:
    raise RuntimeError("Frozen Stage 4A row-order verification failed.")
if not stage4b_row_order_valid:
    raise RuntimeError("Frozen Stage 4B row-order verification failed.")
if not stage4a_stage4b_row_order_alignment_verified:
    raise RuntimeError("Stage 4A and Stage 4B row-order alignment failed.")
if not stage4a_stage4b_rcv_alignment_verified:
    raise RuntimeError("Stage 4A and Stage 4B RCV-key alignment failed.")

# Recency formula
recency_days_t0 = pd.to_numeric(
    t0_raw["recency_days"], errors="coerce"
).astype(float)
saved_recency_t0 = pd.to_numeric(
    t0["recency_score"], errors="coerce"
).astype(float)

expected_recency_t0 = (
    1.0 - recency_days_t0 / MAXIMUM_OBSERVATION_WINDOW_DAYS
).clip(0.0, 1.0)
expected_recency_t0.loc[recency_days_t0.isna()] = np.nan

recency_formula_mismatches = int((~np.isclose(
    saved_recency_t0.to_numpy(),
    expected_recency_t0.to_numpy(),
    rtol=1e-12,
    atol=1e-12,
    equal_nan=True,
)).sum())

# Missingness formula
saved_missing_stage4a_t0 = pd.to_numeric(
    t0_raw["recency_missing_flag"], errors="coerce"
).astype(float)
saved_missing_stage4b_t0 = pd.to_numeric(
    t0["recency_missing_flag"], errors="coerce"
).astype(float)
expected_missing_t0 = recency_days_t0.isna().astype(float)

recency_missing_stage4a_mismatches = int((~np.isclose(
    saved_missing_stage4a_t0.to_numpy(),
    expected_missing_t0.to_numpy(),
    rtol=0.0,
    atol=0.0,
    equal_nan=True,
)).sum())

recency_missing_stage4b_mismatches = int((~np.isclose(
    saved_missing_stage4b_t0.to_numpy(),
    expected_missing_t0.to_numpy(),
    rtol=0.0,
    atol=0.0,
    equal_nan=True,
)).sum())

recency_missing_mismatches = (
    recency_missing_stage4a_mismatches
    + recency_missing_stage4b_mismatches
)

# Submitter formula
submitter_count_t0 = pd.to_numeric(
    t0_raw["unique_submitter_count"], errors="raise"
).astype(float)
expected_log_submitter_t0 = np.log1p(submitter_count_t0)
saved_log_submitter_t0 = pd.to_numeric(
    t0_raw["log1p_unique_submitter_count"], errors="raise"
).astype(float)

log_submitter_mismatches = int((~np.isclose(
    saved_log_submitter_t0.to_numpy(),
    expected_log_submitter_t0.to_numpy(),
    rtol=1e-12,
    atol=1e-12,
)).sum())

observed_log_min = float(expected_log_submitter_t0.min())
observed_log_max = float(expected_log_submitter_t0.max())

expected_submitter_score_t0 = (
    (expected_log_submitter_t0 - SUBMITTER_LOG_MIN)
    / (SUBMITTER_LOG_MAX - SUBMITTER_LOG_MIN)
).clip(0.0, 1.0)

saved_submitter_score_t0 = pd.to_numeric(
    t0["submitter_diversity_score"], errors="raise"
).astype(float)

submitter_formula_mismatches = int((~np.isclose(
    saved_submitter_score_t0.to_numpy(),
    expected_submitter_score_t0.to_numpy(),
    rtol=1e-12,
    atol=1e-12,
)).sum())

# Review-confidence formula
expected_review_t0 = pd.to_numeric(
    t0_raw["aggregate_review_stars"], errors="raise"
).astype(float)
saved_review_t0 = pd.to_numeric(
    t0["review_confidence"], errors="raise"
).astype(float)

review_formula_mismatches = int((~np.isclose(
    saved_review_t0.to_numpy(),
    expected_review_t0.to_numpy(),
    rtol=0.0,
    atol=0.0,
)).sum())

formula_audit = pd.DataFrame([
    {
        "transformation": "recency_score",
        "rows_audited": len(t0),
        "mismatch_count": recency_formula_mismatches,
        "exactly_reproduced": recency_formula_mismatches == 0,
    },
    {
        "transformation": "recency_missing_flag_stage4a",
        "rows_audited": len(t0_raw),
        "mismatch_count": recency_missing_stage4a_mismatches,
        "exactly_reproduced": recency_missing_stage4a_mismatches == 0,
    },
    {
        "transformation": "recency_missing_flag_stage4b",
        "rows_audited": len(t0),
        "mismatch_count": recency_missing_stage4b_mismatches,
        "exactly_reproduced": recency_missing_stage4b_mismatches == 0,
    },
    {
        "transformation": "log1p_unique_submitter_count",
        "rows_audited": len(t0_raw),
        "mismatch_count": log_submitter_mismatches,
        "exactly_reproduced": log_submitter_mismatches == 0,
    },
    {
        "transformation": "submitter_diversity_score",
        "rows_audited": len(t0),
        "mismatch_count": submitter_formula_mismatches,
        "exactly_reproduced": submitter_formula_mismatches == 0,
    },
    {
        "transformation": "review_confidence",
        "rows_audited": len(t0),
        "mismatch_count": review_formula_mismatches,
        "exactly_reproduced": review_formula_mismatches == 0,
    },
])


# --------------------------------------------------------------------------------------------------
# 10. LOAD T1 SOURCE AND RECONSTRUCT THE SIX FEATURES IN MEMORY
# --------------------------------------------------------------------------------------------------

t1_schema_columns = pq.ParquetFile(T1_PARQUET).schema_arrow.names

t1_columns = OrderedDict([
    (
        "rcv_accession",
        resolve_column(
            t1_schema_columns,
            ["rcv_accession", "t1_rcv_accession"],
            "T1 RCV accession",
        ),
    ),
    (
        "target_gene",
        resolve_column(
            t1_schema_columns,
            ["target_genes_json", "target_gene", "gene"],
            "T1 target gene",
        ),
    ),
    (
        "classification_axis",
        resolve_column(
            t1_schema_columns,
            ["aggregate_classification_axis", "classification_axis"],
            "T1 classification axis",
        ),
    ),
    (
        "embedded_cutoff",
        resolve_column(
            t1_schema_columns,
            [
                "embedded_data_cutoff_date",
                "data_cutoff_date",
                "embedded_cutoff_date",
            ],
            "T1 embedded cutoff date",
        ),
    ),
    (
        "aggregate_last_evaluated",
        resolve_column(
            t1_schema_columns,
            ["aggregate_last_evaluated", "last_evaluated"],
            "T1 aggregate last evaluated",
        ),
    ),
    (
        "unique_submitter_count",
        resolve_column(
            t1_schema_columns,
            ["unique_submitter_count_xml", "unique_submitter_count"],
            "T1 unique submitter count",
        ),
    ),
    (
        "aggregate_review_stars",
        resolve_column(
            t1_schema_columns,
            ["aggregate_review_stars", "review_stars"],
            "T1 aggregate review stars",
        ),
    ),
    (
        "aggregate_conflict_flag",
        resolve_column(
            t1_schema_columns,
            ["aggregate_conflict_flag", "conflict_flag"],
            "T1 aggregate conflict flag",
        ),
    ),
    (
        "scv_group_counts_json",
        resolve_column(
            t1_schema_columns,
            ["scv_group_counts_json", "group_counts_json"],
            "T1 SCV group counts",
        ),
    ),
    (
        "scv_count",
        resolve_column(
            t1_schema_columns,
            ["scv_count_xml", "scv_count"],
            "T1 SCV count",
        ),
    ),
])

read_columns = list(dict.fromkeys(t1_columns.values()))
t1 = pd.read_parquet(T1_PARQUET, columns=read_columns).copy()

rcv_t1 = (
    t1[t1_columns["rcv_accession"]]
    .astype("string")
    .str.strip()
    .str.upper()
)

gene_t1 = t1[t1_columns["target_gene"]].map(normalize_gene)

axis_t1 = (
    t1[t1_columns["classification_axis"]]
    .astype("string")
    .fillna("NoClassification")
    .str.strip()
    .replace("", "NoClassification")
)

cutoff_t1_timestamp = pd.to_datetime(
    t1[t1_columns["embedded_cutoff"]],
    errors="coerce",
    utc=True,
).dt.tz_convert(None)

cutoff_t1 = cutoff_t1_timestamp.dt.strftime("%Y-%m-%d")

last_evaluated_t1 = pd.to_datetime(
    t1[t1_columns["aggregate_last_evaluated"]],
    errors="coerce",
    utc=True,
).dt.tz_convert(None)

recency_days_t1 = (
    cutoff_t1_timestamp - last_evaluated_t1
).dt.days.astype(float)

recency_score_t1 = (
    1.0 - recency_days_t1 / MAXIMUM_OBSERVATION_WINDOW_DAYS
).clip(0.0, 1.0)
recency_score_t1.loc[last_evaluated_t1.isna()] = np.nan

recency_missing_flag_t1 = last_evaluated_t1.isna().astype(float)

submitter_count_t1 = pd.to_numeric(
    t1[t1_columns["unique_submitter_count"]],
    errors="coerce",
).astype(float)

log_submitter_t1 = np.log1p(submitter_count_t1)

submitter_diversity_score_t1 = (
    (log_submitter_t1 - SUBMITTER_LOG_MIN)
    / (SUBMITTER_LOG_MAX - SUBMITTER_LOG_MIN)
).clip(0.0, 1.0)

review_confidence_t1 = pd.to_numeric(
    t1[t1_columns["aggregate_review_stars"]],
    errors="coerce",
).astype(float)

conflict_t1 = t1[t1_columns["aggregate_conflict_flag"]].map(
    boolish_to_float
).astype(float)

entropy_results = t1[t1_columns["scv_group_counts_json"]].map(
    normalized_entropy_from_group_counts
)

entropy_t1 = pd.Series(
    [item[0] for item in entropy_results],
    index=t1.index,
    dtype=float,
)
group_count_sum_t1 = pd.Series(
    [item[1] for item in entropy_results],
    index=t1.index,
    dtype=float,
)
positive_group_count_t1 = pd.Series(
    [item[2] for item in entropy_results],
    index=t1.index,
    dtype="int64",
)
entropy_status_t1 = pd.Series(
    [item[3] for item in entropy_results],
    index=t1.index,
    dtype="string",
)

scv_count_t1 = pd.to_numeric(
    t1[t1_columns["scv_count"]],
    errors="coerce",
).astype(float)

scv_group_count_mismatch_mask = ~np.isclose(
    group_count_sum_t1.to_numpy(),
    scv_count_t1.to_numpy(),
    rtol=0.0,
    atol=0.0,
    equal_nan=False,
)
scv_group_count_mismatches = int(
    scv_group_count_mismatch_mask.sum()
)

post_cutoff_dates = int(
    (
        last_evaluated_t1.notna()
        & cutoff_t1_timestamp.notna()
        & (last_evaluated_t1 > cutoff_t1_timestamp)
    ).sum()
)

t1_features = pd.DataFrame({
    "recency_score": recency_score_t1,
    "recency_missing_flag": recency_missing_flag_t1,
    "submitter_diversity_score": submitter_diversity_score_t1,
    "review_confidence": review_confidence_t1,
    "aggregate_conflict_flag": conflict_t1,
    "scv_group_entropy_normalized": entropy_t1,
})

# Explicitly prohibit persistence of t1_features in Cell 7A2.


# --------------------------------------------------------------------------------------------------
# 11. VERIFY FROZEN MODEL PACKAGES AND APPLY PREPROCESSING ONLY
# --------------------------------------------------------------------------------------------------

full_artifact = joblib.load(FULL_MODEL_PATH)
no_star_artifact = joblib.load(NO_STAR_MODEL_PATH)

model_jobs = OrderedDict([
    ("full_ges", (full_artifact, FULL_FEATURES)),
    ("no_star_ges", (no_star_artifact, NO_STAR_FEATURES)),
])

model_results = OrderedDict()
model_inventory_rows = []

for model_name, (artifact, expected_features) in model_jobs.items():
    pipeline, pipeline_location = extract_pipeline(artifact, model_name)

    imputer_name, imputer = get_exact_component(pipeline, SimpleImputer)
    scaler_name, scaler = get_exact_component(pipeline, StandardScaler)
    classifier_name, classifier = get_exact_component(
        pipeline, LogisticRegression
    )

    feature_order, feature_order_source, feature_order_verified = (
        find_feature_order(artifact, pipeline, expected_features)
    )

    fitted_state_verified = bool(
        hasattr(imputer, "statistics_")
        and hasattr(scaler, "mean_")
        and hasattr(scaler, "scale_")
        and hasattr(classifier, "coef_")
        and hasattr(classifier, "intercept_")
        and hasattr(classifier, "classes_")
    )

    settings_verified = model_settings_match(classifier)

    input_frame = t1_features[expected_features].copy()

    transformed = preprocess_without_scoring(
        pipeline,
        input_frame,
    )

    transformed_shape = np.asarray(
        transformed.toarray() if sparse.issparse(transformed) else transformed
    ).shape

    result = {
        "model": model_name,
        "pipeline_location": pipeline_location,
        "pipeline_steps": [name for name, _ in pipeline.steps],
        "imputer_step": imputer_name,
        "scaler_step": scaler_name,
        "classifier_step": classifier_name,
        "feature_order": feature_order,
        "feature_order_source": feature_order_source,
        "feature_order_verified": bool(feature_order_verified),
        "fitted_state_verified": fitted_state_verified,
        "settings_verified": settings_verified,
        "rows": int(transformed_shape[0]),
        "columns": int(transformed_shape[1]),
        "all_finite": transformed_values_are_finite(transformed),
        "classifier_scoring_applied": False,
    }

    model_results[model_name] = result

    model_inventory_rows.append({
        "model": model_name,
        "artifact_path": str(
            FULL_MODEL_PATH
            if model_name == "full_ges"
            else NO_STAR_MODEL_PATH
        ),
        "artifact_sha256": sha256_file(
            FULL_MODEL_PATH
            if model_name == "full_ges"
            else NO_STAR_MODEL_PATH
        ),
        "pipeline_location": pipeline_location,
        "pipeline_steps": json.dumps(result["pipeline_steps"]),
        "feature_order_source": feature_order_source,
        "expected_feature_order": json.dumps(expected_features),
        "observed_feature_order": json.dumps(feature_order),
        "feature_order_verified": feature_order_verified,
        "fitted_state_verified": fitted_state_verified,
        "settings_verified": settings_verified,
        "t1_preprocessing_rows": result["rows"],
        "t1_preprocessing_columns": result["columns"],
        "t1_preprocessing_all_finite": result["all_finite"],
        "classifier_scoring_applied": False,
    })

model_inventory = pd.DataFrame(model_inventory_rows)


# --------------------------------------------------------------------------------------------------
# 12. BUILD FEATURE AND CLASSIFICATION-AXIS INVENTORIES
# --------------------------------------------------------------------------------------------------

feature_inventory_rows = []

for feature in FULL_FEATURES:
    t0_values = pd.to_numeric(t0[feature], errors="coerce").astype(float)
    t1_values = pd.to_numeric(t1_features[feature], errors="coerce").astype(float)

    t0_nonmissing = t0_values.dropna()
    t1_nonmissing = t1_values.dropna()

    t0_min = float(t0_nonmissing.min()) if len(t0_nonmissing) else np.nan
    t0_max = float(t0_nonmissing.max()) if len(t0_nonmissing) else np.nan
    domain_upper = 4.0 if feature == "review_confidence" else 1.0

    feature_inventory_rows.append({
        "feature": feature,
        "used_by_full_ges": feature in FULL_FEATURES,
        "used_by_no_star_ges": feature in NO_STAR_FEATURES,
        "t0_rows": int(len(t0_values)),
        "t0_missing": int(t0_values.isna().sum()),
        "t0_min": t0_min,
        "t0_max": t0_max,
        "t1_rows": int(len(t1_values)),
        "t1_missing": int(t1_values.isna().sum()),
        "t1_min": (
            float(t1_nonmissing.min()) if len(t1_nonmissing) else np.nan
        ),
        "t1_max": (
            float(t1_nonmissing.max()) if len(t1_nonmissing) else np.nan
        ),
        "t1_below_t0_range": int(
            (t1_nonmissing < t0_min).sum()
        ) if np.isfinite(t0_min) else 0,
        "t1_above_t0_range": int(
            (t1_nonmissing > t0_max).sum()
        ) if np.isfinite(t0_max) else 0,
        "t1_outside_frozen_domain": int(
            (
                (t1_nonmissing < 0.0)
                | (t1_nonmissing > domain_upper)
            ).sum()
        ),
        "t1_infinite_values": int(
            np.isinf(t1_values.to_numpy()).sum()
        ),
        "frozen_median_imputer_available": True,
        "row_level_feature_persisted": False,
    })

feature_inventory = pd.DataFrame(feature_inventory_rows)

axis_inventory = (
    pd.DataFrame({
        "classification_axis": axis_t1,
        "target_gene": gene_t1,
    })
    .groupby(["classification_axis", "target_gene"], dropna=False)
    .size()
    .reset_index(name="rows")
)


def axis_policy(axis):
    if axis == "GermlineClassification":
        return "primary_germline_domain_candidate"
    if axis == "NoClassification":
        return "retain_as_evidence_only_not_as_classification"
    return "retain_separately_requires_prespecified_non_germline_policy"


axis_inventory["scientific_applicability"] = (
    axis_inventory["classification_axis"].map(axis_policy)
)
axis_inventory["included_in_preprocessing_check"] = True
axis_inventory["score_created_in_cell_7a2"] = False

observed_gene_counts = {
    str(key): int(value)
    for key, value in gene_t1.value_counts(dropna=False).items()
}

observed_axis_counts = {
    str(key): int(value)
    for key, value in axis_t1.value_counts(dropna=False).items()
}


# --------------------------------------------------------------------------------------------------
# 13. VERIFY THE FROZEN COMBINED-METADATA POLICY WITHOUT APPLYING IT
# --------------------------------------------------------------------------------------------------

stage6a_policy_payload = json.loads(
    STAGE6A_POLICY.read_text(encoding="utf-8")
)
stage6a_policy_text = json.dumps(
    stage6a_policy_payload,
    sort_keys=True,
).lower()

combined_metadata_policy_identified = bool(
    "combined_metadata_instability_risk" in stage6a_policy_text
    and "frozen_before_outcome_label_load_or_temporal_performance"
        in stage6a_policy_text
)


# --------------------------------------------------------------------------------------------------
# 14. IMMUTABILITY RECHECK AND QUALITY-CONTROL REGISTER
# --------------------------------------------------------------------------------------------------

immutable_hashes_after = {
    key: sha256_file(path)
    for key, path in ARTIFACT_PATHS.items()
}
immutable_inputs_unchanged = (
    immutable_hashes_before == immutable_hashes_after
)

entropy_parse_errors = int(
    entropy_status_t1.eq("parse_error").sum()
)
entropy_invalid_values = int(
    entropy_status_t1.eq("invalid_numeric_value").sum()
)
entropy_missing_rows = int(entropy_t1.isna().sum())

all_infinite_feature_values = int(sum(
    np.isinf(
        pd.to_numeric(
            t1_features[column],
            errors="coerce",
        ).to_numpy()
    ).sum()
    for column in t1_features.columns
))

qc_checks = OrderedDict([
    (
        "all_18_frozen_hashes_verified",
        all(
            observed_hashes[key] == EXPECTED_HASHES[key]
            for key in EXPECTED_HASHES
        ),
    ),
    (
        "cell_7a1_manifest_sidecar_verified",
        sidecar_is_valid(CELL_7A1_MANIFEST),
    ),
    (
        "cell_7a1_terminal_decision_verified",
        cell_7a1_payload.get("terminal_decision")
        == EXPECTED_CELL_7A1_DECISION,
    ),
    (
        "cell_7a2_authorized",
        cell_7a1_payload.get(
            "next_authorized_cell", {}
        ).get("cell_id") == "7A2",
    ),
    (
        "t1_dimensions_verified",
        t1_meta.num_rows == EXPECTED_T1_ROWS
        and t1_meta.num_columns == EXPECTED_T1_COLUMNS,
    ),
    (
        "stage4a_dimensions_verified",
        stage4a_meta.num_rows == EXPECTED_STAGE4A_ROWS
        and stage4a_meta.num_columns == EXPECTED_STAGE4A_COLUMNS,
    ),
    (
        "stage4b_dimensions_verified",
        stage4b_meta.num_rows == EXPECTED_STAGE4B_ROWS
        and stage4b_meta.num_columns == EXPECTED_STAGE4B_COLUMNS,
    ),
    ("stage4a_zero_based_row_order_verified", stage4a_row_order_valid),
    ("stage4b_zero_based_row_order_verified", stage4b_row_order_valid),
    (
        "stage4a_stage4b_row_order_alignment_verified",
        stage4a_stage4b_row_order_alignment_verified,
    ),
    (
        "stage4a_stage4b_rcv_alignment_verified",
        stage4a_stage4b_rcv_alignment_verified,
    ),
    ("t0_recency_formula_exact", recency_formula_mismatches == 0),
    (
        "t0_recency_missing_formula_exact",
        recency_missing_mismatches == 0,
    ),
    ("t0_log_submitter_formula_exact", log_submitter_mismatches == 0),
    (
        "t0_submitter_score_formula_exact",
        submitter_formula_mismatches == 0,
    ),
    (
        "t0_review_confidence_formula_exact",
        review_formula_mismatches == 0,
    ),
    (
        "t0_submitter_min_matches_frozen_parameter",
        bool(np.isclose(
            observed_log_min,
            SUBMITTER_LOG_MIN,
            rtol=1e-12,
            atol=1e-12,
        )),
    ),
    (
        "t0_submitter_max_matches_frozen_parameter",
        bool(np.isclose(
            observed_log_max,
            SUBMITTER_LOG_MAX,
            rtol=1e-12,
            atol=1e-12,
        )),
    ),
    ("t1_loaded_rows_verified", len(t1) == EXPECTED_T1_ROWS),
    (
        "t1_rcv_nonmissing",
        int(rcv_t1.isna().sum()) == 0
        and int(rcv_t1.fillna("").eq("").sum()) == 0,
    ),
    (
        "t1_rcv_unique",
        rcv_t1.nunique(dropna=False) == EXPECTED_T1_ROWS,
    ),
    (
        "t1_gene_complete",
        int(gene_t1.fillna("").eq("").sum()) == 0,
    ),
    (
        "t1_gene_counts_verified",
        observed_gene_counts == EXPECTED_T1_GENE_COUNTS,
    ),
    (
        "t1_axis_counts_verified",
        observed_axis_counts == EXPECTED_T1_AXIS_COUNTS,
    ),
    (
        "t1_cutoff_verified",
        set(cutoff_t1.dropna().unique().tolist()) == {"2025-12-27"},
    ),
    ("t1_no_post_cutoff_dates", post_cutoff_dates == 0),
    (
        "t1_submitter_counts_positive",
        int((submitter_count_t1.dropna() <= 0).sum()) == 0,
    ),
    (
        "t1_review_stars_in_valid_clinvar_domain",
        int((
            (review_confidence_t1.dropna() < 0)
            | (review_confidence_t1.dropna() > 4)
        ).sum()) == 0,
    ),
    (
        "t1_conflict_values_complete",
        int(conflict_t1.isna().sum()) == 0,
    ),
    (
        "t1_entropy_json_parse_errors_zero",
        entropy_parse_errors == 0,
    ),
    (
        "t1_entropy_invalid_numeric_values_zero",
        entropy_invalid_values == 0,
    ),
    ("t1_entropy_complete", entropy_missing_rows == 0),
    (
        "t1_entropy_within_domain",
        int(((entropy_t1 < 0) | (entropy_t1 > 1)).sum()) == 0,
    ),
    (
        "t1_group_counts_reconcile_to_scv_count",
        scv_group_count_mismatches == 0,
    ),
    (
        "t1_no_infinite_feature_values",
        all_infinite_feature_values == 0,
    ),
    (
        "full_ges_feature_order_verified",
        model_results["full_ges"]["feature_order_verified"],
    ),
    (
        "full_ges_fitted_state_verified",
        model_results["full_ges"]["fitted_state_verified"],
    ),
    (
        "full_ges_settings_verified",
        model_results["full_ges"]["settings_verified"],
    ),
    (
        "full_ges_preprocessing_shape_verified",
        model_results["full_ges"]["rows"] == EXPECTED_T1_ROWS
        and model_results["full_ges"]["columns"] == len(FULL_FEATURES),
    ),
    (
        "full_ges_preprocessing_all_finite",
        model_results["full_ges"]["all_finite"],
    ),
    (
        "no_star_feature_order_verified",
        model_results["no_star_ges"]["feature_order_verified"],
    ),
    (
        "no_star_fitted_state_verified",
        model_results["no_star_ges"]["fitted_state_verified"],
    ),
    (
        "no_star_settings_verified",
        model_results["no_star_ges"]["settings_verified"],
    ),
    (
        "no_star_preprocessing_shape_verified",
        model_results["no_star_ges"]["rows"] == EXPECTED_T1_ROWS
        and model_results["no_star_ges"]["columns"]
            == len(NO_STAR_FEATURES),
    ),
    (
        "no_star_preprocessing_all_finite",
        model_results["no_star_ges"]["all_finite"],
    ),
    (
        "combined_metadata_policy_identified",
        combined_metadata_policy_identified,
    ),
    ("immutable_inputs_unchanged", immutable_inputs_unchanged),
    ("weak_labels_not_created", True),
    ("model_fit_not_called", True),
    ("model_refit_not_called", True),
    ("predict_not_called", True),
    ("predict_proba_not_called", True),
    ("decision_function_not_called", True),
    ("t1_full_ges_score_not_created", True),
    ("t1_no_star_score_not_created", True),
    ("t1_combined_metadata_score_not_created", True),
    ("threshold_or_weight_optimization_not_performed", True),
    ("row_level_t1_feature_artifact_not_written", True),
    ("rag_corpus_not_constructed", True),
    ("embeddings_not_constructed", True),
    ("question_set_not_constructed", True),
    ("llm_not_called", True),
])

failed_checks = [
    name
    for name, passed in qc_checks.items()
    if passed is not True
]
passed_checks = len(qc_checks) - len(failed_checks)
all_checks_passed = len(failed_checks) == 0

final_decision = (
    "PASS_STAGE7A2_FROZEN_FEATURE_TRANSFORMS_AND_MODEL_PACKAGES_VERIFIED_"
    "T1_FEATURE_RECONSTRUCTION_AND_PREPROCESSING_APPLICABILITY_CONFIRMED_"
    "NO_SCORING_STAGE7A3_FROZEN_T1_SCORE_MATERIALIZATION_AUTHORIZED"
    if all_checks_passed
    else "FAIL_STAGE7A2_PREFLIGHT_STAGE7A3_NOT_AUTHORIZED"
)

# Fail before writing any Cell 7A2 artifact.
if not all_checks_passed:
    print("CELL 7A2 PREFLIGHT FAILED BEFORE OUTPUT FREEZE")
    print("Failed checks:")
    for failed_name in failed_checks:
        print(" -", failed_name)
    raise RuntimeError(
        "Cell 7A2 preflight failed before any Cell 7A2 artifact was written."
    )


# --------------------------------------------------------------------------------------------------
# 15. PRESERVE CREATION TIMESTAMP ACROSS SAFE IDENTICAL RERUNS
# --------------------------------------------------------------------------------------------------

existing_created_utc = None

for candidate in [
    OUTPUTS["manifest"],
    OUTPUTS["preflight_report"],
    OUTPUTS["qc"],
]:
    if candidate.exists():
        try:
            existing_created_utc = json.loads(
                candidate.read_text(encoding="utf-8")
            ).get("created_utc")
            if existing_created_utc:
                break
        except Exception:
            pass

created_utc = (
    existing_created_utc
    or datetime.now(timezone.utc).isoformat()
)


# --------------------------------------------------------------------------------------------------
# 16. WRITE TABLE OUTPUTS
# --------------------------------------------------------------------------------------------------

output_hashes = OrderedDict()

for key, frame in [
    ("artifact_inventory", artifact_inventory),
    ("formula_audit", formula_audit),
    ("feature_inventory", feature_inventory),
    ("model_inventory", model_inventory),
    ("axis_inventory", axis_inventory),
]:
    output_hashes[key] = stable_write_csv(OUTPUTS[key], frame)
    write_sidecar(OUTPUTS[key])

    if not sidecar_is_valid(OUTPUTS[key]):
        raise AssertionError(
            f"Output sidecar verification failed: {OUTPUTS[key]}"
        )


# --------------------------------------------------------------------------------------------------
# 17. WRITE PREFLIGHT AND QC JSON
# --------------------------------------------------------------------------------------------------

preflight_report = {
    "cell_id": CELL_ID,
    "notebook_name": NOTEBOOK_NAME,
    "package_version": PACKAGE_VERSION,
    "created_utc": created_utc,
    "purpose": (
        "Frozen T1 feature reconstruction and model-applicability "
        "preflight without scoring."
    ),
    "authorization": {
        "prior_cell": "7A1",
        "prior_manifest_path": str(CELL_7A1_MANIFEST),
        "prior_manifest_sha256": sha256_file(CELL_7A1_MANIFEST),
        "prior_terminal_decision_verified": True,
        "model_fitting_authorized": False,
        "t1_scoring_authorized_in_cell_7a2": False,
        "threshold_or_weight_optimization_authorized": False,
        "embedding_construction_authorized": False,
        "llm_generation_authorized": False,
    },
    "frozen_inputs": {
        key: {
            "path": str(ARTIFACT_PATHS[key]),
            "sha256": observed_hashes[key],
        }
        for key in ARTIFACT_PATHS
    },
    "t1_source_column_mapping": t1_columns,
    "frozen_transformation_constants": {
        "source_path": str(STAGE4B_TRANSFORM_PARAMETERS),
        "source_sha256": sha256_file(STAGE4B_TRANSFORM_PARAMETERS),
        "maximum_observation_window_days":
            MAXIMUM_OBSERVATION_WINDOW_DAYS,
        "submitter_log_min": SUBMITTER_LOG_MIN,
        "submitter_log_max": SUBMITTER_LOG_MAX,
        "observed_t0_submitter_log_min": observed_log_min,
        "observed_t0_submitter_log_max": observed_log_max,
    },
    "t0_formula_reproduction": {
        row["transformation"]: {
            "rows_audited": int(row["rows_audited"]),
            "mismatch_count": int(row["mismatch_count"]),
            "exactly_reproduced": bool(row["exactly_reproduced"]),
        }
        for row in formula_audit.to_dict("records")
    },
    "t1_accounting": {
        "rows": int(len(t1)),
        "columns": int(t1_meta.num_columns),
        "unique_rcv_accessions": int(rcv_t1.nunique(dropna=False)),
        "gene_counts": observed_gene_counts,
        "classification_axis_counts": observed_axis_counts,
        "embedded_cutoff_values": sorted(
            cutoff_t1.dropna().unique().tolist()
        ),
        "post_cutoff_dates": post_cutoff_dates,
        "entropy_parse_errors": entropy_parse_errors,
        "entropy_invalid_numeric_values": entropy_invalid_values,
        "entropy_missing_rows": entropy_missing_rows,
        "scv_group_count_mismatches": scv_group_count_mismatches,
        "all_infinite_feature_values": all_infinite_feature_values,
    },
    "model_applicability": model_results,
    "combined_metadata_policy": {
        "path": str(STAGE6A_POLICY),
        "sha256": sha256_file(STAGE6A_POLICY),
        "identified": combined_metadata_policy_identified,
        "applied": False,
    },
    "scientific_boundary": {
        "t1_features_reconstructed_in_memory": True,
        "row_level_t1_feature_table_persisted": False,
        "frozen_preprocessing_applied": True,
        "classifier_scoring_applied": False,
        "full_ges_score_created": False,
        "no_star_score_created": False,
        "combined_metadata_score_created": False,
        "weak_labels_created": False,
        "model_fitted": False,
        "model_refitted": False,
        "model_tuned": False,
        "threshold_optimized": False,
        "weight_optimized": False,
        "rag_corpus_constructed": False,
        "embeddings_constructed": False,
        "question_set_constructed": False,
        "llm_called": False,
    },
    "software": {
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "pyarrow": __import__("pyarrow").__version__,
        "scikit_learn": __import__("sklearn").__version__,
        "joblib": joblib.__version__,
    },
}

output_hashes["preflight_report"] = stable_write_json(
    OUTPUTS["preflight_report"],
    preflight_report,
)
write_sidecar(OUTPUTS["preflight_report"])

qc_payload = {
    "cell_id": CELL_ID,
    "notebook_name": NOTEBOOK_NAME,
    "package_version": PACKAGE_VERSION,
    "created_utc": created_utc,
    "total_checks": len(qc_checks),
    "passed_checks": passed_checks,
    "failed_checks": failed_checks,
    "all_checks_passed": all_checks_passed,
    "checks": [
        {
            "check": name,
            "passed": bool(passed),
        }
        for name, passed in qc_checks.items()
    ],
    "decision": final_decision,
}

output_hashes["qc"] = stable_write_json(
    OUTPUTS["qc"],
    qc_payload,
)
write_sidecar(OUTPUTS["qc"])

for key in ["preflight_report", "qc"]:
    if not sidecar_is_valid(OUTPUTS[key]):
        raise AssertionError(
            f"Output sidecar verification failed: {OUTPUTS[key]}"
        )


# --------------------------------------------------------------------------------------------------
# 18. WRITE MANIFEST AND AUTHORIZE CELL 7A3
# --------------------------------------------------------------------------------------------------

output_records = {
    key: {
        "path": str(OUTPUTS[key]),
        "sha256": sha256_file(OUTPUTS[key]),
        "sidecar_path": str(sidecar_path(OUTPUTS[key])),
        "sidecar_sha256": sha256_file(sidecar_path(OUTPUTS[key])),
    }
    for key in OUTPUTS
    if key != "manifest"
}

manifest_payload = {
    "cell_id": CELL_ID,
    "notebook_name": NOTEBOOK_NAME,
    "package_version": PACKAGE_VERSION,
    "created_utc": created_utc,
    "purpose": (
        "Frozen T1 feature reconstruction and model-applicability "
        "preflight without score generation."
    ),
    "upstream_lineage": {
        key: {
            "path": str(path),
            "sha256": observed_hashes[key],
        }
        for key, path in ARTIFACT_PATHS.items()
    },
    "scientific_boundary": {
        "t1_features_reconstructed_in_memory": True,
        "row_level_t1_feature_table_persisted": False,
        "frozen_preprocessing_applied": True,
        "classifier_scoring_applied": False,
        "full_ges_score_created": False,
        "no_star_score_created": False,
        "combined_metadata_score_created": False,
        "weak_labels_created": False,
        "model_fitted": False,
        "model_refitted": False,
        "model_tuned": False,
        "threshold_optimized": False,
        "weight_optimized": False,
        "rag_corpus_constructed": False,
        "embeddings_constructed": False,
        "question_set_constructed": False,
        "llm_called": False,
    },
    "output_artifacts": output_records,
    "qc": {
        "path": str(OUTPUTS["qc"]),
        "sha256": sha256_file(OUTPUTS["qc"]),
        "passed_checks": passed_checks,
        "failed_checks": len(failed_checks),
        "total_checks": len(qc_checks),
    },
    "decision": final_decision,
    "terminal_decision": final_decision,
    "next_authorized_cell": {
        "cell_id": "7A3",
        "name": "Frozen T1 score materialization and checksum freeze",
    },
}

output_hashes["manifest"] = stable_write_json(
    OUTPUTS["manifest"],
    manifest_payload,
)
write_sidecar(OUTPUTS["manifest"])

if not sidecar_is_valid(OUTPUTS["manifest"]):
    raise AssertionError(
        "Cell 7A2 manifest sidecar verification failed."
    )

manifest_readback = json.loads(
    OUTPUTS["manifest"].read_text(encoding="utf-8")
)

if manifest_readback.get("decision") != final_decision:
    raise AssertionError(
        "Cell 7A2 manifest decision readback failed."
    )

for path in OUTPUTS.values():
    if not sidecar_is_valid(path):
        raise AssertionError(
            f"Fresh output verification failed: {path}"
        )


# --------------------------------------------------------------------------------------------------
# 19. CONTROLLED FINAL OUTPUT
# --------------------------------------------------------------------------------------------------

separator = "=" * 144

print("\n" + separator)
print("EXPERIMENT 2 — STAGE 7A — CELL 7A2")
print(
    "FROZEN T1 FEATURE RECONSTRUCTION AND "
    "MODEL-APPLICABILITY PREFLIGHT"
)
print(separator)
print(f"Notebook                                      : {NOTEBOOK_NAME}")
print(f"Project root                                  : {ROOT}")

print("\nUPSTREAM AUTHORIZATION")
print(
    f"Cell 7A1 manifest SHA-256                     : "
    f"{sha256_file(CELL_7A1_MANIFEST)}"
)
print("Cell 7A1 terminal PASS verified               : YES")
print("Model fitting authorized                      : NO")
print("T1 scoring authorized in Cell 7A2             : NO")
print("Embedding construction authorized             : NO")
print("LLM generation authorized                     : NO")

print("\nFROZEN INPUT VERIFICATION")
print(
    f"Frozen artifacts verified                     : "
    f"{len(observed_hashes)}/{len(EXPECTED_HASHES)}"
)
print(
    f"T1 Parquet SHA-256                            : "
    f"{sha256_file(T1_PARQUET)}"
)
print(
    f"Stage 4A feature table SHA-256                 : "
    f"{sha256_file(STAGE4A_FEATURE_TABLE)}"
)
print(
    f"Stage 4B transform table SHA-256               : "
    f"{sha256_file(STAGE4B_TABLE)}"
)
print(
    f"Stage 4C Full-GES model SHA-256                : "
    f"{sha256_file(FULL_MODEL_PATH)}"
)
print(
    f"Stage 4C No-star model SHA-256                 : "
    f"{sha256_file(NO_STAR_MODEL_PATH)}"
)
print(
    f"Stage 6A policy SHA-256                        : "
    f"{sha256_file(STAGE6A_POLICY)}"
)

print("\nFROZEN TRANSFORMATION REPRODUCTION")
print(
    f"Frozen recency window days                    : "
    f"{MAXIMUM_OBSERVATION_WINDOW_DAYS:.17g}"
)
print(
    f"Frozen submitter log minimum                  : "
    f"{SUBMITTER_LOG_MIN:.17g}"
)
print(
    f"Observed T0 submitter log minimum             : "
    f"{observed_log_min:.17g}"
)
print(
    f"Frozen submitter log maximum                  : "
    f"{SUBMITTER_LOG_MAX:.17g}"
)
print(
    f"Observed T0 submitter log maximum             : "
    f"{observed_log_max:.17g}"
)

for row in formula_audit.to_dict("records"):
    print(
        f"{row['transformation']:<46}: "
        f"{int(row['mismatch_count']):,} mismatches"
    )

print("\nT1 FEATURE RECONSTRUCTION — IN MEMORY ONLY")
print(f"T1 rows                                       : {len(t1):,}")
print(
    f"Unique RCV accessions                         : "
    f"{rcv_t1.nunique(dropna=False):,}"
)
print(
    f"Recency missing                               : "
    f"{int(recency_score_t1.isna().sum()):,}"
)
print(f"Entropy missing                               : {entropy_missing_rows:,}")
print(
    f"Entropy JSON parse errors                     : "
    f"{entropy_parse_errors:,}"
)
print(
    f"SCV/group-count mismatches                    : "
    f"{scv_group_count_mismatches:,}"
)
print(f"Post-cutoff dates                             : {post_cutoff_dates:,}")
print(
    f"Infinite derived feature values               : "
    f"{all_infinite_feature_values:,}"
)
print("Row-level reconstructed feature table saved   : NO")

print("\nFROZEN MODEL APPLICABILITY")
for row in model_inventory.to_dict("records"):
    print(
        f"{row['model']:<46}: "
        f"{int(row['t1_preprocessing_rows']):,} rows × "
        f"{int(row['t1_preprocessing_columns'])} features | "
        f"finite={row['t1_preprocessing_all_finite']} | scored=NO"
    )

print("\nCLASSIFICATION-AXIS ACCOUNTING")
for axis in EXPECTED_T1_AXIS_COUNTS:
    print(
        f"{axis:<46}: "
        f"{int(observed_axis_counts.get(axis, 0)):,}"
    )

print("\nCELL 7A2 FROZEN OUTPUTS")
for label, key in [
    ("Frozen artifact inventory", "artifact_inventory"),
    ("T0 transformation formula audit", "formula_audit"),
    ("T1 feature applicability inventory", "feature_inventory"),
    ("Frozen model pipeline inventory", "model_inventory"),
    ("Classification-axis applicability", "axis_inventory"),
    ("Preflight report", "preflight_report"),
    ("QC record", "qc"),
    ("Manifest", "manifest"),
]:
    path = OUTPUTS[key]
    print(f"{label:<46}: {path}")
    print(f"{'SHA-256':<46}: {sha256_file(path)}")

print(
    f"\nQC checks                                      : "
    f"{passed_checks}/{len(qc_checks)} PASS"
)

print("\nSCIENTIFIC OPERATIONS")
print("Full-GES fitted or refitted                    : NO")
print("No-star GES fitted or refitted                 : NO")
print("Full-GES T1 score generated                    : NO")
print("No-star T1 score generated                     : NO")
print("Combined-metadata T1 score generated           : NO")
print("Threshold or weight optimization               : NO")
print("RAG corpus constructed                         : NO")
print("Embeddings constructed                         : NO")
print("LLM called                                     : NO")

print("\nNEXT AUTHORIZED CELL")
print(
    "Cell 7A3                                      : "
    "Frozen T1 Full-GES, No-star-GES,"
)
print(
    "                                                 "
    "and combined-metadata score"
)
print(
    "                                                 "
    "materialization and checksum freeze"
)
print("Model fitting                                 : PROHIBITED")
print("Threshold or weight optimization              : PROHIBITED")
print("Embedding construction                        : PROHIBITED")
print("LLM generation                                : PROHIBITED")

print(f"\nFINAL DECISION                                : {final_decision}")
print(separator)


Mounted at /content/drive

EXPERIMENT 2 — STAGE 7A — CELL 7A2
FROZEN T1 FEATURE RECONSTRUCTION AND MODEL-APPLICABILITY PREFLIGHT
Notebook                                      : 05_GES_Aware_Genomic_RAG.ipynb
Project root                                  : /content/drive/MyDrive/GES_RAG_Temporal_Study

UPSTREAM AUTHORIZATION
Cell 7A1 manifest SHA-256                     : 84e509d97f01fb8dc0b6ad0c7e24de762e8923b9068fc835bf328105f79e946d
Cell 7A1 terminal PASS verified               : YES
Model fitting authorized                      : NO
T1 scoring authorized in Cell 7A2             : NO
Embedding construction authorized             : NO
LLM generation authorized                     : NO

FROZEN INPUT VERIFICATION
Frozen artifacts verified                     : 18/18
T1 Parquet SHA-256                            : 5713a11bdbf4804758cc011f9b2f302afc91fa1f88c1b178d675c28bb277d37c
Stage 4A feature table SHA-256                 : c100b3781e6801425f622f5d091376abfe0939e48c0a792af32f6eebe6401f

## Required stopping rule

Proceed to **Cell 7A3** only when the last code cell ends with:

- every QC check passing,
- the long `PASS_STAGE7A2_...` terminal decision, and
- `Cell 7A3` listed as the next authorized cell.

Do not continue after a failed check.